In [ ]:
# Ячейка 1: Постановка
"""
## Муниципальное здравоохранение

Прямая задача (Primal):
Мэрия распределяет бюджет между поликлиниками (X) и стационарами (Y).
Прибыль (охват пациентов): 200 с поликлиники, 300 со стационара.

Ресурсы:
- Врачи: 2X + 3Y <= 120
- Оборудование: 4X + Y <= 100
- Бюджет: X + 2Y <= 80
X, Y >= 0

Задание:
1. Решить прямую задачу
2. Записать двойственную модель
3. Найти теневые цены (shadow prices)
4. Определить дефицитные ресурсы
5. Проверить прогноз: увеличить врачей на 1
"""

In [ ]:
# Ячейка 2: Решение прямой задачи
import numpy as np
from scipy.optimize import linprog

# Primal: max Z = 200X + 300Y -> min -Z
c_primal = [-200, -300]
A = [[2, 3], [4, 1], [1, 2]]
b = [120, 100, 80]

result_primal = linprog(c_primal, A_ub=A, b_ub=b, bounds=(0, None), method='highs')

X_opt, Y_opt = result_primal.x
Z_opt = -result_primal.fun

print("Прямая задача:")
print(f"Поликлиники X = {X_opt:.2f}")
print(f"Стационары Y = {Y_opt:.2f}")
print(f"Охват Z = {Z_opt:.2f}")

# Запасы ресурсов (slack)
slack = b - A @ result_primal.x
print("\nЗапасы ресурсов:")
resources = ['Врачи', 'Оборудование', 'Бюджет']
for i, (name, s) in enumerate(zip(resources, slack)):
    status = 'дефицит' if s < 0.001 else f'избыток {s:.1f}'
    print(f"{name}: {status}")

In [ ]:
# Ячейка 3: Двойственная задача
"""
Двойственная модель (Dual):

min W = 120y1 + 100y2 + 80y3

Ограничения:
2y1 + 4y2 + y3 >= 200  (поликлиники)
3y1 + y2 + 2y3 >= 300  (стационары)
y1, y2, y3 >= 0

y1 - теневая цена врачей
y2 - теневая цена оборудования
y3 - теневая цена бюджета
"""

# Решаем dual
c_dual = [120, 100, 80]
A_dual = [[-2, -4, -1], [-3, -1, -2]]  # >= в <=
b_dual = [-200, -300]

result_dual = linprog(c_dual, A_ub=A_dual, b_ub=b_dual, bounds=(0, None), method='highs')

print("Двойственная задача (теневые цены):")
print(f"y1 (врачи) = {result_dual.x[0]:.2f}")
print(f"y2 (оборудование) = {result_dual.x[1]:.2f}")
print(f"y3 (бюджет) = {result_dual.x[2]:.2f}")
print(f"W = {result_dual.fun:.2f}")

print(f"\nПроверка: Z = W ? {Z_opt:.2f} = {result_dual.fun:.2f}")

In [ ]:
# Ячейка 4: Проверка прогноза
print("Прогноз: +1 врач -> охват вырастет на y1 =", result_dual.x[0])

# Повторное решение с b1 = 121
b_new = [121, 100, 80]
result_new = linprog(c_primal, A_ub=A, b_ub=b_new, bounds=(0, None), method='highs')
delta = -result_new.fun - Z_opt

print(f"Реальное изменение: {delta:.4f}")
print(f"Совпадает с теневой ценой: {abs(delta - result_dual.x[0]) < 0.01}")

In [ ]:
# Ячейка 5: Вывод
"""
Активные ограничения: врачи и бюджет (запас = 0).
Неактивное: оборудование (избыток).
Теневые цены показывают реальную ценность ресурсов.
"""